In [9]:
# -*- coding: utf-8 -*-
# 【CP1-06 并行归约】扇出-扇入与字段并发写入的归约规则
# 文件：CP1/06_node_state.ipynb
# 作用：本 Cell 演示 【CP1-06 并行归约】扇出-扇入与字段并发写入的归约规则 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

""" TYPED DICT """
from typing import TypedDict,Annotated
from operator import add
from langgraph.types import Overwrite
from langgraph.graph import StateGraph,START,END

class OverAllState(TypedDict):
    # 规约的方式是追加合并
    logs: Annotated[list[str], add]
    # 如果出现并行节点，往下游传递节点，必须要有reducer函数
    cur_id: Annotated[str, add]  # 并行写入时以拼接归约，演示用途

# -------- 并行归约详解 --------
# 本图扇出：node_1 -> {node_2, node_3} 并行 -> node_4 汇合
#   - logs: 带 add，并行写入通过“多版本并发归约”汇总，安全
#   - cur_id: 若无 reducer，两分支的写入会竞争，框架取“最后完成者”的值（非确定）
#     本例为演示，将 cur_id 也设为 Annotated[str, add]，结果为字符串拼接
#     生产中对单值并行写入应避免：要么改写不同字段，要么显式设计归约

def node_1(state:OverAllState) -> OverAllState:
    """node_1节点"""
    print("node_1节点运行")
    for k,v in state.items():
        print(k,v)
    return {
        "cur_id":  "node_1",
        "logs": ["node_1 运行完毕"]
    }

def node_2(state:OverAllState) -> OverAllState:
    """node_2节点"""
    print("node_2节点运行")
    for k,v in state.items():
        print(k,v)
    return {
        "cur_id":  "node_2",
        "logs": ["node_2 运行完毕"]
    }

from time import sleep

def node_3(state:OverAllState) -> OverAllState:
    """node_3节点"""
    sleep(1)
    """node_3节点"""
    print("node_3节点运行")
    for k,v in state.items():
        print(k,v)
    return {
        "cur_id":  "node_3",
        "logs": ["node_3 运行完毕"]
    }

def node_4(state:OverAllState) -> OverAllState:
    """node_4节点"""
    sleep(2)
    print("node_4节点运行")
    for k,v in state.items():
        print(k,v)
    return {
        "cur_id":  "node_4",
        "logs": ["node_4 运行完毕"]
    }
    
builder = StateGraph(state_schema=OverAllState)  # 创建状态图构建器：绑定状态 Schema

builder.add_node(node_1)  # 注册节点到图中
builder.add_node(node_2)  # 注册节点到图中
builder.add_node(node_3)  # 注册节点到图中
builder.add_node(node_4)  # 注册节点到图中

builder.add_edge(START,"node_1")  # 起点扇出
builder.add_edge("node_1","node_2")
builder.add_edge("node_1","node_3")
builder.add_edge("node_2","node_4")
builder.add_edge("node_3","node_4")
builder.add_edge("node_4",END)  # 汇入终点

graph=builder.compile()  # 编译图：蓝图→可执行对象

result = graph.invoke({"logs":["start"],"cur_id":"start"})  # 触发图执行：传入初始 State + config
print(result)


node_1节点运行
logs ['start']
cur_id start
node_2节点运行
logs ['start', 'node_1 运行完毕']
cur_id startnode_1
node_3节点运行
logs ['start', 'node_1 运行完毕']
cur_id startnode_1
node_4节点运行
logs ['start', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕']
cur_id startnode_1node_2node_3
{'logs': ['start', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕', 'node_4 运行完毕'], 'cur_id': 'startnode_1node_2node_3node_4'}
